Save as: module7-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-07/module7-simulation-lab.ipynb)

# Lab 7 — Synthetic Cultural Agents
### Calibrating a population, and grading the calibration
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Can a demographically conditioned LLM agent
population reproduce the *level*, *shape*, and *demographic gradients*
of human dictator-game giving in two different cultural contexts — and
which uses does the achieved calibration rung actually license?

**The design difference from every prior lab:** agents are **sampled,
not fixed cells.** Each agent draws a demographic profile — age,
education, income, religiosity, community type — from context-specific
distributions (the SCA method: persona conditioning on survey-derived
marginals rather than hand-written stereotypes). Two contexts by
default: US urban and rural Kenya, 60 agents each, playing a \$10
dictator game.

**The calibration hierarchy (state which rung your claims need):**
1. Marginals match the survey (built in — proves nothing about behavior)
2. A behavioral moment matches the literature (necessary, not sufficient)
3. Gradients & heterogeneity match (usually fails first)
4. Treatment responses match (what a pilot needs — unverifiable from
   inside the sim)

**Benchmarks:** you must enter calibration targets **with citations,
before running** — the notebook ships with placeholders marked
"(verify)". Sources: Engel (2011) meta-analysis; the Henrich et al.
cross-cultural program.

**Hypotheses to state before running (see handout):** H1 levels within
±5 pp of your cited targets; H2 the contexts *differ* in the
human-documented direction (the expected failure: convergence to ~30%
everywhere — anglophone training data flattening culture); H3 one named
demographic gradient per context, with sign.

**How to run:** no prior Python experience needed. Run the cells top to
bottom. Your required modification (handout) goes **only** in the cells
marked **CHANGE HERE**.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## Population design — CHANGE HERE (all five blocks live in this cell)

Each context defines sampling distributions for five demographic
attributes (values with weights). `sample_profile()` draws one agent.
The `TARGETS` dict is your pre-registered calibration benchmark — cite
the paper, replace the "(verify)" placeholders, and do it BEFORE
running.

Your required modification will usually mean **(1)** adding a third
context or a new attribute (e.g. market integration) to `CONTEXTS`,
**(2)** updating `TARGETS` with a cited benchmark, **(3)** weaving new
attributes into `build_prompt` *without changing other wording*, and
**(4)** touching `parse_response` / `mock_response` only if the task or
answer format changes (the ultimatum modification changes both).


In [ ]:
CONTEXTS = {                                     # CHANGE HERE (1 of 5)
    "us_urban": {
        "country": "the United States",
        "community": [("a large city", 1.0)],
        "age": [(25, 0.3), (35, 0.3), (45, 0.2), (60, 0.2)],
        "education": [("a high-school education", 0.35),
                      ("a college degree", 0.45),
                      ("a graduate degree", 0.20)],
        "income": [("a low household income", 0.3),
                   ("a middle household income", 0.5),
                   ("a high household income", 0.2)],
        "religiosity": [("not religious", 0.35),
                        ("somewhat religious", 0.40),
                        ("deeply religious", 0.25)],
    },
    "kenya_rural": {
        "country": "Kenya",
        "community": [("a rural village", 1.0)],
        "age": [(25, 0.35), (35, 0.3), (45, 0.2), (60, 0.15)],
        "education": [("a primary-school education", 0.45),
                      ("some secondary schooling", 0.35),
                      ("a completed secondary education", 0.20)],
        "income": [("income from subsistence farming", 0.6),
                   ("income from a small family business", 0.4)],
        "religiosity": [("somewhat religious", 0.35),
                        ("deeply religious", 0.60),
                        ("not religious", 0.05)],
    },
}

# Pre-registered calibration targets: (mean share given, citation).
# REPLACE the placeholders with published estimates for YOUR contexts,
# and cite them — this is part of the graded write-up.  CHANGE HERE (2 of 5)
TARGETS = {
    "us_urban": (0.28, "Engel (2011) meta-analytic mean giving (verify "
                       "against the student/adult subsample you intend)"),
    "kenya_rural": (0.35, "(verify — replace with the published estimate "
                          "for your target population, e.g. from the "
                          "Henrich et al. cross-cultural program)"),
}

ENDOWMENT = 10
N_AGENTS = 60          # sampled agents per context


def sample_profile(context: str) -> dict:
    """Draw one agent's demographic profile from the context marginals."""
    spec = CONTEXTS[context]
    profile = {"context": context, "country": spec["country"]}
    for attr in ["community", "age", "education", "income", "religiosity"]:
        values, weights = zip(*spec[attr])
        profile[attr] = random.choices(values, weights=weights, k=1)[0]
    return profile


def build_prompt(profile: dict) -> str:          # CHANGE HERE (3 of 5)
    """Dictator game, persona conditioned on the sampled profile.
    Design rules: demographics first, task second, response format last;
    neutral task wording (no 'fair'/'generous'); identical task text
    across contexts — only the profile sentence varies."""
    return (
        f"You are a {profile['age']}-year-old adult living in "
        f"{profile['community']} in {profile['country']}. You have "
        f"{profile['education']}, your household relies on "
        f"{profile['income']}, and you are {profile['religiosity']}.\n\n"
        "You are taking part in a paid decision study run by a local "
        f"university. You have been given ${ENDOWMENT}. You may give any "
        "whole-dollar amount between $0 and "
        f"${ENDOWMENT} to another anonymous participant from your "
        "community. You keep the rest. Decide as you genuinely would.\n\n"
        'Respond with ONLY a JSON object: {"give": <integer 0-10>}.'
    )


def parse_response(text: str) -> "dict | None":  # CHANGE HERE (4 of 5)
    """Extract the amount given. Return None on failure — never guess."""
    match = re.search(r'\{\s*"give"\s*:\s*(\d+)\s*\}', text)
    if match:
        give = int(match.group(1))
        if 0 <= give <= ENDOWMENT:
            return {"give": give}
    return None


def mock_response(prompt: str) -> str:           # CHANGE HERE (5 of 5)
    """DRY_RUN only: synthetic placeholder answers so the notebook
    executes without API calls. NOT model behavior — these numbers are
    made up and deliberately produce legible calibration patterns
    (context level differences, human-like piles at 0 and 5, mild
    demographic gradients). Edit only if the task or format changes."""
    center = 3.5 if "Kenya" in prompt else 2.8
    if "deeply religious" in prompt:
        center += 0.5
    if "60-year-old" in prompt or "45-year-old" in prompt:
        center += 0.3
    r = random.random()
    if r < 0.12:
        give = 0                                  # the selfish pile
    elif r < 0.35:
        give = 5                                  # the equal-split pile
    else:
        give = int(round(min(max(random.gauss(center, 1.5), 0), 10)))
    return json.dumps({"give": give})


## Run the experiment *(do not modify)*

Samples `N_AGENTS` demographic profiles per context, queries each agent
once, logs parse failures instead of dropping them, saves
`lab7_results.csv` (one row per agent, demographics included — the
gradient checks need them) and `lab7_prompts_log.json` (three example
prompts per context, for the appendix). With `DRY_RUN = False` this
makes ~120 API calls.


In [ ]:
def run_experiment() -> pd.DataFrame:
    rows, example_prompts = [], {}
    total = len(CONTEXTS) * N_AGENTS
    print(f"{len(CONTEXTS)} contexts x {N_AGENTS} sampled agents = {total} calls")

    for context in CONTEXTS:
        example_prompts[context] = []
        for i in range(N_AGENTS):
            profile = sample_profile(context)
            prompt = build_prompt(profile)
            if i < 3:
                example_prompts[context].append(prompt)
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=100,
                    temperature=TEMPERATURE,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = response.content[0].text
                decision = parse_response(raw)
            except Exception as err:            # log failures; never drop silently
                raw, decision = f"ERROR: {err}", None
                time.sleep(5)
            rows.append({
                **profile, "rep": i,
                "give": None if decision is None else decision["give"],
                "raw": raw, "model": MODEL, "temperature": TEMPERATURE,
            })
        done = [r for r in rows if r["give"] is not None and r["context"] == context]
        print(f"  context {context} done ({len(done)}/{N_AGENTS} parsed)")

    df = pd.DataFrame(rows)
    df.to_csv("lab7_results.csv", index=False)
    # Example prompts — required in the write-up appendix.
    with open("lab7_prompts_log.json", "w") as f:
        json.dump(example_prompts, f, indent=2)
    return df


df = run_experiment()


## The calibration report *(do not modify)*

Three checks, one per rung above the built-in marginals:
**(1) Level** — mean share given vs. your cited target, per context.
**(2) Shape** — the offer distribution: human data piles at \$0, \$2–3,
and \$5; a single tight spike is a red flag.
**(3) Gradients** — giving by religiosity and age within context: flat
gradients mean the demographics never mattered (the context label did
all the work).


In [ ]:
valid = df.dropna(subset=["give"]).copy()
valid["share"] = valid["give"] / ENDOWMENT
print(f"Parse-failure rate: {df['give'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

print("\n=== (1) LEVEL: mean share given vs. cited target ===")
for context, (target, cite) in TARGETS.items():
    sub = valid[valid["context"] == context]["share"]
    if len(sub):
        print(f"  {context}: silicon {sub.mean():.3f}  |  target {target:.3f}"
              f"  |  gap {abs(sub.mean() - target) * 100:.1f} pp")
        print(f"      target source: {cite}")

print("\n=== (2) SHAPE: distribution of amounts given ===")
shape = valid.pivot_table(index="give", columns="context",
                          values="rep", aggfunc="count").fillna(0).astype(int)
print(shape)
print("Human data piles at $0, $2-3, and $5. One tight spike = red flag.")

print("\n=== (3) GRADIENTS: does giving vary with demographics? ===")
for attr in ["religiosity", "age"]:
    print(f"\n  Mean share given by {attr}:")
    print(valid.pivot_table(index=attr, columns="context",
                            values="share", aggfunc="mean").round(3))
print("\nCompare signs to the human literature you cited (e.g. giving "
      "rising with religiosity and age). Flat gradients = the context "
      "label, not the population, produced your level match.")

print("\n=== Cross-context variation ===")
means = valid.groupby("context")["share"].mean()
print(means.round(3))
print("If the contexts converge (~0.30 everywhere), you are looking at "
      "WEIRD flattening — diagnose it in Section 5; it IS a finding.")


## Robustness — required in every lab

Rerun at least 30 agents per context with:

1. **a paraphrased prompt** (rewrite `build_prompt`'s wording, same
   content);
2. **a second model** (change `MODEL` in the setup cell);
3. **the context-blinding run** — remove the country name from the
   prompt, keep every demographic attribute. If behavior doesn't
   change, your calibration was the label, not the population; if it
   collapses to one distribution, the demographics were never doing
   the work.

**Limits of silicon subjects:** three bite hardest here. (1) A matched
mean can be *recitation* — the meta-analyses are training data, and an
exact match should raise suspicion, not confidence. (2) Thin personas
drift WEIRD — the model defaults to its training distribution's
center, flattening exactly the cross-cultural variation the population
exists to capture. (3) Rung 4 of the calibration hierarchy —
treatment-response calibration, the rung an effect-size pilot needs —
is circular from inside the sim: verifying it requires the human data
the pilot was meant to replace. Your write-up's Section 5 places the
population on the hierarchy and defends the placement; Section 6 states
which uses that rung honestly licenses (PS7 Part C is the exam version
of this).
